In [2]:
!git clone https://github.com/GomathimeenaM/GL_MML.git

Cloning into 'GL_MML'...
remote: Enumerating objects: 540, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 540 (delta 0), reused 1 (delta 0), pack-reused 536 (from 1)
Receiving objects: 100% (540/540), 309.77 MiB | 29.13 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (500/500), done.


In [ ]:
# Install
!pip install sklearn-crfsuite librosa opencv-python scikit-learn

In [ ]:
import os
import numpy as np
import cv2
import librosa
import sklearn_crfsuite
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

In [ ]:
# -----------------------------
# PATHS
# -----------------------------
FACE_DIR = "GL_MML/Unit_4/face_images"
VOICE_DIR = "GL_MML/Unit_4/voices_for_images"


In [ ]:
# -----------------------------
# LOAD DATA
# -----------------------------
def load_data(face_dir, voice_dir):
    face_files = sorted(os.listdir(face_dir))
    voice_files = sorted(os.listdir(voice_dir))

    data = []
    labels = []

    for f in face_files:
        name = f.split('.')[0]
        voice_file = name + ".wav"

        if voice_file in voice_files:
            face_path = os.path.join(face_dir, f)
            voice_path = os.path.join(voice_dir, voice_file)

            label = name.split('_')[0]

            data.append((face_path, voice_path))
            labels.append(label)

    return data, labels

data, labels = load_data(FACE_DIR, VOICE_DIR)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(labels)

In [ ]:
# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_face(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    return img.flatten()

def extract_voice(path):
    y, sr = librosa.load(path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfcc, axis=1)

face_feat = []
voice_feat = []

for fpath, vpath in data:
    face_feat.append(extract_face(fpath))
    voice_feat.append(extract_voice(vpath))

face_feat = np.array(face_feat)
voice_feat = np.array(voice_feat)

# Reduce dimension
pca = PCA(n_components=9)
face_feat = pca.fit_transform(face_feat)

In [ ]:
# -----------------------------
# CREATE MULTI-VIEW FEATURES
# -----------------------------
def create_features(face_feat, voice_feat):
    X = []
    for i in range(len(face_feat)):
        feat = {
            "face_mean": float(np.mean(face_feat[i])),
            "face_std": float(np.std(face_feat[i])),
            "voice_mean": float(np.mean(voice_feat[i])),
            "voice_std": float(np.std(voice_feat[i]))
        }
        X.append([feat])
    return X

X = create_features(face_feat, voice_feat)

# Labels format for CRF
y = [[label] for label in labels]


In [ ]:
# -----------------------------
# TRAIN CRF
# -----------------------------
crf = sklearn_crfsuite.CRF(max_iterations=100)
crf.fit(X, y)

# -----------------------------
# PREDICT
# -----------------------------
pred = crf.predict(X)

print("\nPredictions:")
print(pred)

# Compare
print("\nActual vs Predicted:")
for i in range(len(labels)):
    print(labels[i], "->", pred[i][0])